# Closed Deals - Silver Transformation

## Parameters

In [0]:
dbutils.widgets.text(
    name="environment",
    defaultValue="dev",
    label="Environment"
)

environment = dbutils.widgets.get("environment").strip().lower()

if environment not in ("dev", "test", "prod"):
    raise ValueError(
        f"Unsupported environment: {environment}. Expected dev, test, or prod."
    )

## Setup

In [0]:
from pyspark.sql.functions import col

catalog = f"ecommerce_{environment}"
source_table = f"{catalog}.bronze.olist_closed_deals"
target_table = f"{catalog}.silver.olist_closed_deals"

## Read Bronze data

In [0]:
bronze_df = spark.table(source_table)

## Transform to Silver

In [0]:
silver_df = (
    bronze_df
    .withColumnRenamed(
        "mql_id",
        "marketing_qualified_lead_id"
    )
    .withColumnRenamed(
        "sdr_id",
        "sales_development_representative_id"
    )
    .withColumnRenamed(
        "sr_id",
        "sales_representative_id"
    )
    .withColumnRenamed(
        "won_date",
        "deal_won_date"
    )
    .withColumnRenamed(
        "has_gtin",
        "has_global_trade_item_number"
    )
)

In [0]:
silver_df = silver_df.withColumn(
    "declared_product_catalog_size",
    col("declared_product_catalog_size").cast("int")
)

## Write to Silver

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)